In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, Dense
from sklearn.metrics import mean_squared_error
from hmmlearn.hmm import GaussianHMM
from sklearn.preprocessing import StandardScaler

In [2]:
DATA_PATH = "data\\new_data\\Shanghai_T1DM_merged.xlsx"

df = pd.read_excel(DATA_PATH)


In [3]:
df.head()

,Date,CGM (mg / dl),CBG (mg / dl),Blood Ketone (mmol / L),Dietary intake,饮食,Insulin dose - s.c.,Non-insulin hypoglycemic agents,"CSII - bolus insulin (Novolin R, IU)","CSII - basal insulin (Novolin R, IU / H)",Insulin dose - i.v.,Patient_ID
0,2021-07-30 16:43:00,113.4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.3,NaN,1001_0_20210730
1,2021-07-30 16:58:00,124.2,NaN,NaN,NaN,NaN,NaN,NaN,4.0,NaN,NaN,1001_0_20210730
2,2021-07-30 17:13:00,129.6,NaN,NaN,data not available,未记录,NaN,NaN,NaN,NaN,NaN,1001_0_20210730
3,2021-07-30 17:28:00,142.2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1001_0_20210730
4,2021-07-30 17:43:00,156.6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1001_0_20210730


In [4]:
df.describe()

,Date,CGM (mg / dl),CBG (mg / dl),Blood Ketone (mmol / L),"CSII - bolus insulin (Novolin R, IU)","CSII - basal insulin (Novolin R, IU / H)",Insulin dose - i.v.
count,4368,4368.000000,124.000000,0.0,29.000000,44.000000,0.0
mean,2021-03-11 18:49:47.129120768,188.489835,206.027419,NaN,5.275862,0.454545,NaN
min,2021-01-14 18:39:00,39.600000,45.000000,NaN,2.000000,0.200000,NaN
25%,2021-01-26 03:35:15,127.800000,153.900000,NaN,4.000000,0.300000,NaN
50%,2021-02-19 10:12:30,190.800000,202.500000,NaN,5.000000,0.500000,NaN
75%,2021-03-10 22:11:45,243.000000,266.400000,NaN,6.000000,0.500000,NaN
max,2021-08-06 12:58:00,475.200000,396.000000,NaN,10.000000,0.600000,NaN
std,NaN,74.547343,78.826748,NaN,2.169538,0.128415,NaN


In [5]:
df = df[['Date', 'CGM (mg / dl)', 'Patient_ID']]


In [6]:
df.head()

,Date,CGM (mg / dl),Patient_ID
0,2021-07-30 16:43:00,113.4,1001_0_20210730
1,2021-07-30 16:58:00,124.2,1001_0_20210730
2,2021-07-30 17:13:00,129.6,1001_0_20210730
3,2021-07-30 17:28:00,142.2,1001_0_20210730
4,2021-07-30 17:43:00,156.6,1001_0_20210730


In [7]:
df.describe()

,Date,CGM (mg / dl)
count,4368,4368.000000
mean,2021-03-11 18:49:47.129120768,188.489835
min,2021-01-14 18:39:00,39.600000
25%,2021-01-26 03:35:15,127.800000
50%,2021-02-19 10:12:30,190.800000
75%,2021-03-10 22:11:45,243.000000
max,2021-08-06 12:58:00,475.200000
std,NaN,74.547343


In [8]:
df = df.rename(columns={
    'Date': 'time',
    'CGM (mg / dl)': 'glucose'
})


In [9]:
df.head()

,time,glucose,Patient_ID
0,2021-07-30 16:43:00,113.4,1001_0_20210730
1,2021-07-30 16:58:00,124.2,1001_0_20210730
2,2021-07-30 17:13:00,129.6,1001_0_20210730
3,2021-07-30 17:28:00,142.2,1001_0_20210730
4,2021-07-30 17:43:00,156.6,1001_0_20210730


In [10]:
df['time'] = pd.to_datetime(df['time'])
df['glucose'] = pd.to_numeric(df['glucose'], errors='coerce')


In [11]:
df = df.dropna(subset=['time', 'glucose'])
df = df[df['glucose'] > 0]


In [12]:
df = df.sort_values(['Patient_ID', 'time'])
df = df.reset_index(drop=True)


In [13]:
patient_id = df['Patient_ID'].unique()[0]
patient_df = df[df['Patient_ID'] == patient_id].copy()


In [14]:
patient_df['delta_minutes'] = patient_df['time'].diff().dt.total_seconds() / 60
patient_df['delta_minutes'].describe()


count    657.0
mean      15.0
std        0.0
min       15.0
25%       15.0
50%       15.0
75%       15.0
max       15.0
Name: delta_minutes, dtype: float64

In [15]:
MAX_GAP = 30  # minutes

patient_df = patient_df[
    (patient_df['delta_minutes'].isna()) |
    (patient_df['delta_minutes'] <= MAX_GAP)
]


In [16]:
glucose_values = patient_df['glucose'].values.reshape(-1, 1)


In [17]:
df.to_csv('Shanghai_T1DM_merged_cleaned.csv', index=False)


### Train–test split (time-aware)

In [18]:
train_size = int(len(glucose_values) * 0.8)

train_data = glucose_values[:train_size]
test_data  = glucose_values[train_size:]


### Scale the data (NN ONLY)

In [19]:
scaler = MinMaxScaler()

train_scaled = scaler.fit_transform(train_data)
test_scaled  = scaler.transform(test_data)


In [20]:
def create_sequences(data, window_size):
    X, y = [], []
    for i in range(len(data) - window_size):
        X.append(data[i:i+window_size])
        y.append(data[i+window_size])
    return np.array(X), np.array(y)


In [21]:
WINDOW_SIZE = 12  # e.g. past 1 hour if 5-min CGM


In [22]:
X_train, y_train = create_sequences(train_scaled, WINDOW_SIZE)
X_test,  y_test  = create_sequences(test_scaled, WINDOW_SIZE)


In [23]:
lstm_model = Sequential([
    LSTM(64, input_shape=(WINDOW_SIZE, 1)),
    Dense(1)
])

lstm_model.compile(
    optimizer='adam',
    loss='mse'
)

lstm_model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)


Epoch 1/30


C:\Users\pc\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 24ms/step - loss: 0.1962 - val_loss: 0.0398
Epoch 2/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0274 - val_loss: 0.0261
Epoch 3/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0197 - val_loss: 0.0265
Epoch 4/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0169 - val_loss: 0.0249
Epoch 5/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0154 - val_loss: 0.0247
Epoch 6/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0150 - val_loss: 0.0235
Epoch 7/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0144 - val_loss: 0.0230
Epoch 8/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0140 - val_loss: 0.0216
Epoch 9/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0135 - val_loss: 0.0212
Epoch 10/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0132 - val_loss: 0.0197
Epoch 11/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0132 - val_loss: 0.0209
Epoch 12/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0124 - val_loss: 0.0175

In [24]:
lstm_pred_scaled = lstm_model.predict(X_test)

lstm_pred = scaler.inverse_transform(lstm_pred_scaled)
y_test_true = scaler.inverse_transform(y_test)

lstm_rmse = np.sqrt(mean_squared_error(y_test_true, lstm_pred))
print("LSTM RMSE (mg/dL):", lstm_rmse)


4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
LSTM RMSE (mg/dL): 14.255079186971052


In [25]:
gru_model = Sequential([
    GRU(64, input_shape=(WINDOW_SIZE, 1)),
    Dense(1)
])

gru_model.compile(
    optimizer='adam',
    loss='mse'
)

gru_model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=32,
    validation_split=0.1,
    verbose=1
)


Epoch 1/30


C:\Users\pc\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - loss: 0.1190 - val_loss: 0.0248
Epoch 2/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0197 - val_loss: 0.0164
Epoch 3/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0111 - val_loss: 0.0148
Epoch 4/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0091 - val_loss: 0.0152
Epoch 5/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0082 - val_loss: 0.0125
Epoch 6/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0076 - val_loss: 0.0117
Epoch 7/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0070 - val_loss: 0.0104
Epoch 8/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0065 - val_loss: 0.0096
Epoch 9/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0061 - val_loss: 0.0085
Epoch 10/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0057 - val_loss: 0.0074
Epoch 11/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0053 - val_loss: 0.0067
Epoch 12/30
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.0050 - val_loss: 0.0063

In [26]:
gru_pred_scaled = gru_model.predict(X_test)

gru_pred = scaler.inverse_transform(gru_pred_scaled)

gru_rmse = np.sqrt(mean_squared_error(y_test_true, gru_pred))
print("GRU RMSE (mg/dL):", gru_rmse)


4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
GRU RMSE (mg/dL): 10.882050052552529


In [27]:
scaler_hmm = StandardScaler()
glucose_std = scaler_hmm.fit_transform(glucose_values)


In [28]:
train_size = int(len(glucose_std) * 0.8)

train_hmm = glucose_std[:train_size]
test_hmm  = glucose_std[train_size:]


In [29]:
hmm_model = GaussianHMM(
    n_components=4,          # number of hidden states
    covariance_type="diag",
    n_iter=300,
    random_state=42
)

hmm_model.fit(train_hmm)


GaussianHMM(n_components=4, n_iter=300, random_state=42)

In [30]:
hidden_states = hmm_model.predict(test_hmm)


In [31]:
state_means = hmm_model.means_.flatten()

hmm_pred_std = np.array([state_means[state] for state in hidden_states])


In [32]:
hmm_pred = scaler_hmm.inverse_transform(hmm_pred_std.reshape(-1, 1))
hmm_true = scaler_hmm.inverse_transform(test_hmm)


In [33]:
hmm_rmse = np.sqrt(mean_squared_error(hmm_true, hmm_pred))
print("HMM RMSE (mg/dL):", hmm_rmse)


HMM RMSE (mg/dL): 14.652406769577917


In [35]:
# Naive persistence baseline
naive_pred = test_hmm[:-1]
naive_true = test_hmm[1:]

naive_pred = scaler_hmm.inverse_transform(naive_pred)
naive_true = scaler_hmm.inverse_transform(naive_true)

naive_rmse = np.sqrt(mean_squared_error(naive_true, naive_pred))
print("Naive RMSE (mg/dL):", naive_rmse)


Naive RMSE (mg/dL): 7.789070440085306


In [37]:
print("LSTM RMSE (mg/dL):", lstm_rmse)
print("GRU RMSE (mg/dL):", gru_rmse)
print("HMM RMSE (mg/dL):", hmm_rmse)
print("Naive RMSE (mg/dL):", naive_rmse)

LSTM RMSE (mg/dL): 14.255079186971052
GRU RMSE (mg/dL): 10.882050052552529
HMM RMSE (mg/dL): 14.652406769577917
Naive RMSE (mg/dL): 7.789070440085306


In [39]:
mean_glucose = np.mean(glucose_values)

In [41]:
lstm_acc = 100 * (1 - lstm_rmse / mean_glucose)
gru_acc  = 100 * (1 - gru_rmse / mean_glucose)
hmm_acc  = 100 * (1 - hmm_rmse / mean_glucose)
naive_acc = 100 * (1 - naive_rmse / mean_glucose)
print(f"LSTM Accuracy: {lstm_acc:.2f}%")
print(f"GRU Accuracy: {gru_acc:.2f}%")
print(f"HMM Accuracy: {hmm_acc:.2f}%")
print(f"Naive Accuracy: {naive_acc:.2f}%")

LSTM Accuracy: 92.76%
GRU Accuracy: 94.47%
HMM Accuracy: 92.56%
Naive Accuracy: 96.04%
